In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("desafio_06_agregacoes")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 16:47:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [45]:
df = spark.read.csv("/workspace/datasets/06_agregacoes/vendas.csv", header= True, inferSchema= True)

In [46]:
df_resumo_estado = (
    df.groupBy("estado")
    .agg(
        F.count("venda_id").alias("quantidade_vendas"),
        F.round(F.sum("valor_total")).alias("faturamento_total"),
        F.round(F.avg("valor_total")).alias("ticket_medio"),
        F.countDistinct("vendedor_id").alias("quantidade_vendedores"),
        F.sum(F.when(F.col("forma_pagamento") == "Pix", 1).otherwise(0)).alias("quantidade_vendas_pix"),
        F.round(F.sum(F.when(F.col("forma_pagamento") == "Pix", F.col("valor_total")).otherwise(0))).alias("faturamento_pix")
    )
)

In [47]:
df_resumo_estado.show()

+------+-----------------+-----------------+------------+---------------------+---------------------+---------------+
|estado|quantidade_vendas|faturamento_total|ticket_medio|quantidade_vendedores|quantidade_vendas_pix|faturamento_pix|
+------+-----------------+-----------------+------------+---------------------+---------------------+---------------+
|    GO|              526|        1173497.0|      2231.0|                   20|                  172|       412101.0|
|    SP|             1990|        4492085.0|      2257.0|                   20|                  578|      1315186.0|
|    RS|              541|        1238967.0|      2290.0|                   20|                  164|       381641.0|
|    MG|             1208|        2772609.0|      2295.0|                   20|                  322|       720719.0|
|    BA|             1866|        4296269.0|      2302.0|                   20|                  575|      1292723.0|
|    PE|              608|        1389735.0|      2286.0

In [33]:
df_pagamento_estado = (
    df.groupBy("estado", "forma_pagamento")
    .agg(
        F.count("venda_id").alias("quantidade_vendas"),
        F.round(F.sum("valor_total")).alias("faturamento_total"),
        F.round(F.avg("valor_total")).alias("ticket_medio")
    )
)

In [40]:
df_resumo_global = (
    df.agg(
        F.count("venda_id").alias("quantidade_total_vendas"),
        F.round(F.sum("valor_total")).alias("faturamento_total"),
        F.round(F.avg("valor_total")).alias("ticket_medio_total"),
        F.countDistinct("vendedor_id").alias("quantidade_vendedores_distintos")
    )
)

In [43]:
df_resumo_estado.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, count('venda_id) AS quantidade_vendas#651, round(sum('valor_total), 0) AS faturamento_total#653, round(avg('valor_total), 0) AS ticket_medio#655, count('vendedor_id) AS quantidade_vendedores#657, sum(CASE WHEN ('forma_pagamento = Pix) THEN 1 ELSE 0 END) AS quantidade_vendas_pix#659, round(sum(CASE WHEN ('forma_pagamento = Pix) THEN 'valor_total ELSE 0 END), 0) AS faturamento_pix#661]
+- Relation [venda_id#56,data_venda#57,vendedor_id#58,produto#59,categoria#60,quantidade#61,valor_unitario#62,valor_total#63,cidade#64,estado#65,forma_pagamento#66] csv

== Analyzed Logical Plan ==
estado: string, quantidade_vendas: bigint, faturamento_total: double, ticket_medio: double, quantidade_vendedores: bigint, quantidade_vendas_pix: bigint, faturamento_pix: double
Aggregate [estado#65], [estado#65, count(venda_id#56) AS quantidade_vendas#651L, round(sum(valor_total#63), 0) AS faturamento_total#653, round(avg(valor_total#63), 0) AS ticket_me

1. O que o FileScan está fazendo?

Resposta: FileScan csv representa a leitura física do CSV pelo Spark, incluindo as colunas necessárias para executar a consulta.

2. O que o HashAggregate parcial está fazendo?

Resposta: O hash parcial tem como função atribuir as metricas (count, sum, avg...) apenas dentro das partições, economizando memoria e processamento para o spark la na frente.

3. Por que existe Exchange / Shuffle?

Resposta: O exchange percebe que há a necessidade de redistribuição desses dados, pela chave o shuffle ja é a execução em si, então é como se o spark pegasse todos os dados de cada partição e juntasse eles, o shuffle geralmente ocorre em operações wide

4. Qual chave está sendo usada na redistribuição?

Resposta: Estamos utilizando a chave estado na redistribuição

5. O que o HashAggregate final faz?

Resposta: O hash final faz a atribuição final das metricas das partições (count, sum, avg...), o hash parcial tem grande influencia aqui, pois como havia dito, ele faz as metricas parciais de cada partição, se não fosse por ele o hash final teria que atribuir as metricas de todas as partições sozinhas, o hash parcial economiza bastante processamento do spark.

6. Por que várias métricas não exigem um Shuffle separado para cada uma?

O spark ja consegue executar o shuffle englobando todas as metricas



In [44]:
df_resumo_global.explain(True)

== Parsed Logical Plan ==
'Aggregate [count('venda_id) AS quantidade_total_vendas#1129, round(sum('valor_total), 0) AS faturamento_total#1131, round(avg('valor_total), 0) AS ticket_medio_total#1133, 'count(distinct 'vendedor_id) AS quantidade_vendedores_distintos#1134]
+- Relation [venda_id#778,data_venda#779,vendedor_id#780,produto#781,categoria#782,quantidade#783,valor_unitario#784,valor_total#785,cidade#786,estado#787,forma_pagamento#788] csv

== Analyzed Logical Plan ==
quantidade_total_vendas: bigint, faturamento_total: double, ticket_medio_total: double, quantidade_vendedores_distintos: bigint
Aggregate [count(venda_id#778) AS quantidade_total_vendas#1129L, round(sum(valor_total#785), 0) AS faturamento_total#1131, round(avg(valor_total#785), 0) AS ticket_medio_total#1133, count(distinct vendedor_id#780) AS quantidade_vendedores_distintos#1134L]
+- Relation [venda_id#778,data_venda#779,vendedor_id#780,produto#781,categoria#782,quantidade#783,valor_unitario#784,valor_total#785,cida

Por que nesse caso aparece SinglePartition em vez de hashpartitioning(estado, ...)?

Resposta: Como não estamos nos baseado em um chave, a agregação é feita de forma global, o spark utiliza o singlepartition para executar as metricas.

Por que seria incorreto calcular countDistinct(vendedor_id)
separadamente em cada partição e simplesmente somar
os resultados?

Resposta: Calcular separadamente em cada partição e somar os resultados seria incorreto pois o mesmo vendendor pode aparecer em multiplas partições.